# Edit distance search
Manipulating inference results, creating training and testing labels

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from tqdm import tqdm
import os

In [3]:
basepath = r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\edit_distance_outputs"

In [4]:
def clean_prediction(edit_motif_search_prediction):

    cleaned_prediction = []
    for i in edit_motif_search_prediction:
        if i == 'fake':
            cleaned_prediction.append([])
        else:
            cleaned_prediction.append([int(i[1])])

    return cleaned_prediction

In [5]:

def get_edit_dataframe(edit_inference_filename):

    with open(edit_inference_filename, 'r') as f:
        lines = f.readlines()

    motif_predictions = []
    orientations = []
    read_ids = []
    ont_barcode = []

    for line in tqdm(lines):
        split_line = line.split()
        read_id = split_line[0][3:]
        orientation = split_line[1]
        prediction = split_line[4][8:].split('->')
        if not (prediction[0].startswith('f') or prediction[0].startswith('m')):
            prediction = prediction[1:]

        cleaned_prediction = clean_prediction(prediction)
        
        motif_predictions.append(cleaned_prediction)
        orientations.append(orientation)
        read_ids.append(read_id)
    
    df = pd.DataFrame({"read_id": read_ids, "orientation": orientations, "motif_seq": motif_predictions})
    df = df.drop_duplicates(subset=['read_id'])
    return df

In [6]:
master_df = pd.DataFrame()

for file in tqdm(os.listdir(basepath)):
    df = get_edit_dataframe(os.path.join(basepath, file))
    master_df = pd.concat([master_df, df])

100%|██████████| 235/235 [00:11<00:00, 20.84it/s]


In [10]:
master_df.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\01-04run\edit_medium.pkl")

## Balancing edit-train df

In [5]:
from data_functions import get_cleaned_encoded_file

In [6]:
encoded_df = pd.read_csv(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\EIC01-01-1280-T1_encoded.tsv", sep='\t')

In [136]:
t = get_cleaned_encoded_file(encoded_df)

In [137]:
t = t[['ONT_Barcode', 'HW_Address', 'payload']]

In [9]:
t

,ONT_Barcode,HW_Address,payload
0,1,barcode_external01_internal01,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,..."
1,1,barcode_external02_internal01,"[[3, 4, 7, 8], [2, 3, 4, 8], [2, 4, 6, 7], [1,..."
2,1,barcode_external03_internal01,"[[1, 3, 5, 8], [2, 4, 5, 6], [1, 5, 6, 8], [1,..."
3,1,barcode_external04_internal01,"[[1, 4, 5, 8], [3, 4, 7, 8], [1, 4, 7, 8], [1,..."
4,1,barcode_external05_internal01,"[[2, 4, 5, 6], [3, 4, 6, 7], [4, 5, 6, 8], [1,..."
...,...,...,...
1275,77,barcode_external04_internal08,"[[1, 2, 3, 6], [2, 4, 5, 7], [1, 3, 4, 7], [1,..."
1276,77,barcode_external05_internal08,"[[4, 6, 7, 8], [1, 2, 4, 8], [3, 4, 5, 7], [2,..."
1277,77,barcode_external06_internal08,"[[1, 4, 7, 8], [4, 6, 7, 8], [1, 2, 4, 8], [2,..."
1278,77,barcode_external07_internal08,"[[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4], [1,..."


In [10]:
edit_train_df = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\full_datasets\edit_master_train.pkl")

In [11]:
edit_train_df.head()

,read_id,ONT_Barcode,HW_Address,orientation,start_end,library_motif,squiggle,motif_seq,strand,payload_motifs_found,edit_spacer_seq,edit_motifs_found,payload
0,0117ec74-7ef1-4e8c-b169-c8ca9a576de4,barcode01,barcode_external01_internal01,+|+|+,75-124|370-419|566-615,ltm8_2x1|ltm8_5x4|ltm8_9x1,"[452, 439, 450, 444, 451, 454, 471, 448, 418, ...","[10, 1, 10, 13, 4, 13, 17, 1, 17]",+,3,"[10, 1, 10, 11, 1, 11, 12, 3, 12, 13, 1, 13, 1...",9.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,..."
1,0117ec74-7ef1-4e8c-b169-c8ca9a576de4,barcode01,barcode_external01_internal01,+|+|+,75-124|370-419|566-615,ltm8_2x1|ltm8_5x4|ltm8_9x1,"[452, 439, 450, 444, 451, 454, 471, 448, 418, ...","[10, 1, 10, 13, 4, 13, 17, 1, 17]",+,3,"[10, 1, 10, 11, 1, 11, 12, 3, 12, 13, 1, 13, 1...",9.0,"[[3, 6, 7, 8], [3, 4, 5, 8], [1, 2, 4, 6], [1,..."
2,0117ec74-7ef1-4e8c-b169-c8ca9a576de4,barcode01,barcode_external01_internal01,+|+|+,75-124|370-419|566-615,ltm8_2x1|ltm8_5x4|ltm8_9x1,"[452, 439, 450, 444, 451, 454, 471, 448, 418, ...","[10, 1, 10, 13, 4, 13, 17, 1, 17]",+,3,"[10, 1, 10, 11, 1, 11, 12, 3, 12, 13, 1, 13, 1...",9.0,"[[1, 2, 4, 5], [2, 4, 5, 6], [2, 5, 6, 8], [2,..."
3,0117ec74-7ef1-4e8c-b169-c8ca9a576de4,barcode01,barcode_external01_internal01,+|+|+,75-124|370-419|566-615,ltm8_2x1|ltm8_5x4|ltm8_9x1,"[452, 439, 450, 444, 451, 454, 471, 448, 418, ...","[10, 1, 10, 13, 4, 13, 17, 1, 17]",+,3,"[10, 1, 10, 11, 1, 11, 12, 3, 12, 13, 1, 13, 1...",9.0,"[[2, 3, 4, 8], [3, 5, 6, 7], [2, 3, 7, 8], [2,..."
4,0117ec74-7ef1-4e8c-b169-c8ca9a576de4,barcode01,barcode_external01_internal01,+|+|+,75-124|370-419|566-615,ltm8_2x1|ltm8_5x4|ltm8_9x1,"[452, 439, 450, 444, 451, 454, 471, 448, 418, ...","[10, 1, 10, 13, 4, 13, 17, 1, 17]",+,3,"[10, 1, 10, 11, 1, 11, 12, 3, 12, 13, 1, 13, 1...",9.0,"[[1, 4, 5, 6], [1, 5, 6, 8], [1, 3, 6, 8], [2,..."


In [12]:
edit_train_df = edit_train_df.drop(columns=['payload'])

In [13]:
edit_train_df['ONT_Barcode'] = edit_train_df['ONT_Barcode'].apply(lambda x: int(x[-2:]))

In [138]:
merged_df = pd.merge(filtered_df, t, on=['ONT_Barcode', 'HW_Address'])

Steps
1. Filter out reads where more than 8 motifs are identified
2. Keep the ones with no error
3. For the ones with error, check motif search label and encoded to make the perfect label with no errors
4. Double check that there are no errors for the whole dataset
5. Keep the address motifs identified
6. Some data of the distribution of reads

In [30]:
filtered_df = merged_df.loc[merged_df['edit_motifs_found'] > 9]

In [19]:
from transcript_sorting import sort_transcript_with_address
from utils import evaluate_prediction, create_spacer_sequence_with_address

In [142]:
filtered_df['edit_payload_seq'] = filtered_df['edit_spacer_seq'].apply(lambda x: sort_transcript_with_address(x))

C:\Users\Parv\AppData\Local\Temp\ipykernel_16804\3654313105.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['edit_payload_seq'] = filtered_df['edit_spacer_seq'].apply(lambda x: sort_transcript_with_address(x))


In [33]:
def remove_erroneous_motifs(prediction, original):
    corrected = [[] for i in range(8)]

    cycle_num = 0
    for i, j in zip(prediction, original):
        for k in i:
            if k in j:
                corrected[cycle_num].append(k)
        cycle_num += 1
    return corrected

In [72]:
from tqdm import tqdm

In [74]:
filtered_df.columns

Index(['read_id', 'ONT_Barcode', 'HW_Address', 'orientation', 'start_end',
       'library_motif', 'squiggle', 'motif_seq', 'strand',
       'payload_motifs_found', 'edit_spacer_seq', 'edit_motifs_found',
       'payload', 'edit_payload_seq'],
      dtype='object')

In [141]:
merged_df.columns

Index(['read_id', 'ONT_Barcode', 'HW_Address', 'orientation', 'start_end',
       'library_motif', 'squiggle', 'motif_seq', 'strand',
       'payload_motifs_found', 'edit_spacer_seq', 'edit_motifs_found',
       'payload_x', 'payload_y'],
      dtype='object')

In [145]:
counter = 0
mf = 0
me = 0
corrected_edit_payload_seq = []
unique_predictions = set()

edit_seqs = []
spacer_seqs = []
read_ids = []
payloads = []
squiggles = []
orientation = []

for ind, row in tqdm(filtered_df.iterrows(), total=len(filtered_df)):
    i = row['edit_payload_seq']
    j = row['payload']
    k = row['edit_spacer_seq']
    read_id = row['read_id']



    if str(i) not in unique_predictions:
        metrics = evaluate_prediction(i[2:], j)
        if metrics[1] > 0:
            continue
        mf += metrics[0]
        me += metrics[1]
        read_ids.append(read_id)
        edit_seqs.append(i)
        spacer_seqs.append(k)
        unique_predictions.add(str(i))
    #final = i[:2] + corrected
    #print(i)
    #print(final)
    #print()
    #corrected_edit_payload_seq.append(final)

100%|██████████| 110274/110274 [00:06<00:00, 17521.52it/s]


In [149]:
filtered_df = filtered_df.loc[filtered_df['read_id'].isin(read_ids)]

In [150]:
len(filtered_df)

90534

In [103]:
forward_df = filtered_df_.loc[filtered_df_['read_id'].isin(read_ids)]

In [105]:
forward_df = forward_df.loc[forward_df['orientation'].str.startswith('+')]

In [153]:
filtered_df.to_pickle(r'C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_train_filtered_reverse.pkl')

In [81]:
len(merged_unique_df)

2936700

In [68]:
t = [str(i) for i in filtered_df['edit_payload_seq'].to_list()[:2942440]]

In [157]:
filtered_reverse = filtered_df.loc[filtered_df['orientation'].str.startswith('-')]

In [160]:
filtered_reverse['edit_spacer_seq'] = filtered_reverse['edit_spacer_seq'].apply(lambda x: x[::-1])

C:\Users\Parv\AppData\Local\Temp\ipykernel_4900\2773630298.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_reverse['edit_spacer_seq'] = filtered_reverse['edit_spacer_seq'].apply(lambda x: x[::-1])


In [ ]:
filtered_forward['edit_spacer_seq'] = filtered_df['edit_payload_seq_no_error'].apply()

In [166]:
filtered_reverse.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_train_filtered_reverse.pkl")

In [167]:
filtered_forward = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_train_filtered_forward.pkl")

### Adding edit labels to test dataset

In [3]:
edit_df = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_distance_motif_search.pkl")

In [4]:
edit_df

,read_id,orientation,motif_seq
0,7f177924-d279-4de6-bb86-01eaec9ee0e5,-,"[[3], [5], [2], [3], [], [], [], [], [], []]"
1,d47257cc-0209-450d-8b15-8c99a82dd1c4,-,"[[1], [8], [2], [5], [1], [5], [], [], [], []]"
2,6bf40594-e818-401a-a7f4-142b51615a14,-,"[[4], [2], [2], [1], [], [], [], [], [], []]"
3,07126fb4-14ef-4d7c-8be0-491c44e4ea8a,+,"[[1], [6], [6], [4], [], [], [], [], [], [8]]"
4,ccbd4335-1c81-4c06-a030-de9b231b0eed,-,"[[8], [2], [8], [2], [7], [8], [3], [2], [], []]"
...,...,...,...
590132,a3d4d1f1-5317-489a-b58e-548bf008b914,+,"[[], [], [4], [2], [5], [8], [7], [7], [6], [8]]"
590133,3c753ca5-9b01-4e70-9d67-527c53846aa0,-,"[[], [], [], [], [5], [], [], [], [], []]"
590134,7032acb2-5612-412e-8f2b-18d4be8eab13,-,"[[8], [5], [3], [5], [4], [8], [5], [5], [7], []]"
590135,84c36d62-3594-492e-9fae-b17a12944b54,+,"[[2], [5], [3], [3], [1], [7], [1], [5], [8], ..."
